# Rough-output investigation: different-patient vs same-patient fine-tuning

This is an **evaluation-only** companion to `256_128model_Train-Copy1.ipynb`.
This notebook compares: (1) rigid input, (2) zero-flow Haar reconstruction,
(3) curriculum-only weights, (4) the old different-patient fine-tuning, and
(5) the new same-patient longitudinal fine-tuning. All methods use the same
Haar + VoxelMorph inference path. The goal is to investigate why the old
output appeared rough; model weights must not be averaged.

Run this notebook only after the model and hyperparameters are frozen. TestData
must not be used to change the model. The original training notebook is not
modified, and this notebook creates no optimizer and performs no training.


In [ ]:
import os
import sys
from pathlib import Path

# Select the local custom PyTorch VoxelMorph before importing voxelmorph.
os.environ["VXM_BACKEND"] = "pytorch"

SAITO_ROOT = Path(r"C:\Users\ri0151fv\Saito")
PROJECT_ROOT = Path(r"C:\Users\ri0151fv\OneDrive - 学校法人立命館\ドキュメント\New project")
TEST_ROOT = SAITO_ROOT / "Data" / "TestData"
LONGITUDINAL_ROOT = SAITO_ROOT / "Data" / "Longitudinal22"
PRETRAIN_PATH = SAITO_ROOT / "model_analysis_pipeline_pretrain.pth"
DIFFERENT_PATIENT_PATH = SAITO_ROOT / "model_analysis_pipeline_128_256_256.pth"
FINETUNED_PATH = SAITO_ROOT / "model_analysis_pipeline_longitudinal22_best.pth"
OUTPUT_ROOT = SAITO_ROOT / "Evaluation_Longitudinal22"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

os.chdir(SAITO_ROOT)
for path in (SAITO_ROOT, PROJECT_ROOT):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import csv
import json
import math
import re
from dataclasses import dataclass

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import wilcoxon
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import voxelmorph as vxm

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
torch.manual_seed(20260717)
np.random.seed(20260717)

print("Working directory:", Path.cwd())
print("VoxelMorph:", vxm.__file__)
print("Device:", device)
print("Curriculum-pretrained checkpoint:", PRETRAIN_PATH)
print("Old different-patient fine-tuned checkpoint:", DIFFERENT_PATIENT_PATH)
print("Curriculum + longitudinal fine-tuned checkpoint:", FINETUNED_PATH)


In [ ]:
# Same Haar analysis/downsample/warp/upsample/synthesis path as Copy1.
NB_FEATURES = [
    [32, 64, 64, 64, 64],
    [64, 64, 64, 64, 64, 32, 16, 16],
]
MODEL_SHAPE = (64, 128, 128)
EXPECTED_VOLUME_SHAPE = (128, 256, 256)
GRAD_WEIGHT = 1e-2


class Haar3DAnalysisOnly(nn.Module):
    def __init__(self):
        super().__init__()
        low = torch.tensor([1.0, 1.0], dtype=torch.float32) / math.sqrt(2.0)
        high = torch.tensor([1.0, -1.0], dtype=torch.float32) / math.sqrt(2.0)
        filters = []
        names = []
        for z_name, z_filter in zip(("L", "H"), (low, high)):
            for y_name, y_filter in zip(("L", "H"), (low, high)):
                for x_name, x_filter in zip(("L", "H"), (low, high)):
                    filters.append(
                        z_filter[:, None, None]
                        * y_filter[None, :, None]
                        * x_filter[None, None, :]
                    )
                    names.append(z_name + y_name + x_name)
        self.register_buffer("weight", torch.stack(filters).unsqueeze(1))
        self.names = names

    def forward(self, image):
        image = F.pad(image, (0, 1, 0, 1, 0, 1))
        return F.conv3d(image, self.weight, stride=1, padding=0)


def down_sampling_3d(bands):
    return bands[:, :, ::2, ::2, ::2]


def up_sampling_3d(bands):
    batch, channels, depth, height, width = bands.shape
    upsampled = torch.zeros(
        batch,
        channels,
        depth * 2,
        height * 2,
        width * 2,
        dtype=bands.dtype,
        device=bands.device,
    )
    upsampled[:, :, ::2, ::2, ::2] = bands
    return upsampled


def make_3d_filter(z_filter, y_filter, x_filter):
    return z_filter[:, None, None] * y_filter[None, :, None] * x_filter[None, None, :]


def create_synthesis_filters(target_device):
    low = torch.tensor(
        [1.0, 1.0], dtype=torch.float32, device=target_device
    ) / math.sqrt(2.0)
    high = torch.tensor(
        [1.0, -1.0], dtype=torch.float32, device=target_device
    ) / math.sqrt(2.0)
    filters = torch.stack(
        [
            make_3d_filter(low, low, low),
            make_3d_filter(low, low, high),
            make_3d_filter(low, high, low),
            make_3d_filter(low, high, high),
            make_3d_filter(high, low, low),
            make_3d_filter(high, low, high),
            make_3d_filter(high, high, low),
            make_3d_filter(high, high, high),
        ]
    )
    return torch.flip(filters, dims=(1, 2, 3)).unsqueeze(1)


def synthesis_filter_3d(upsampled, filters):
    _, channel_count, depth, height, width = upsampled.shape
    filtered = []
    for band_index in range(channel_count):
        band = upsampled[:, band_index : band_index + 1]
        result = F.conv3d(
            band, filters[band_index : band_index + 1], stride=1, padding=1
        )
        filtered.append(result[:, :, :depth, :height, :width])
    filtered = torch.cat(filtered, dim=1)
    return torch.sum(filtered, dim=1, keepdim=True), filtered


analysis = Haar3DAnalysisOnly().to(device).eval()
synthesis_filters = create_synthesis_filters(device)
flow_gradient_loss = vxm.losses.Grad("l2").loss


def load_plain_state_dict(path):
    try:
        state = torch.load(path, map_location="cpu", weights_only=True)
    except TypeError:
        state = torch.load(path, map_location="cpu")
    if isinstance(state, dict) and "model_state_dict" in state:
        state = state["model_state_dict"]
    return state


def build_model(checkpoint_path):
    model = vxm.networks.VxmDense_128_256_256(
        MODEL_SHAPE, NB_FEATURES, int_steps=0
    ).to(device)
    model.load_state_dict(load_plain_state_dict(checkpoint_path), strict=True)
    return model.eval()


def registration_forward(model, spatial_transformer, moving, fixed):
    moving_bands = down_sampling_3d(analysis(moving))
    fixed_bands = down_sampling_3d(analysis(fixed))
    flow = model(moving_bands, fixed_bands)
    warped_bands = [
        spatial_transformer(moving_bands[:, index : index + 1], flow)
        for index in range(moving_bands.shape[1])
    ]
    warped = up_sampling_3d(torch.cat(warped_bands, dim=1))
    transformed, _ = synthesis_filter_3d(warped, synthesis_filters)
    return transformed, flow


In [ ]:
PATIENT_RE = re.compile(r"MIC\d+", re.IGNORECASE)
DATE_RE = re.compile(r"_(\d{4})_(\d{4})\.npz$", re.IGNORECASE)


@dataclass(frozen=True)
class TestPair:
    pair_number: int
    patient_id: str
    moving_path: Path
    fixed_path: Path


def natural_pair_number(path):
    match = re.fullmatch(r"pair(\d+)", path.name, re.IGNORECASE)
    if match is None:
        raise ValueError(f"Unexpected TestData directory: {path}")
    return int(match.group(1))


def patient_id_from_name(path):
    match = PATIENT_RE.search(path.name)
    if match is None:
        raise ValueError(f"Patient ID missing from {path}")
    return match.group(0).upper()


def date_from_name(path):
    match = DATE_RE.search(path.name)
    if match is None:
        raise ValueError(f"Date missing from {path}")
    year, month_day = match.groups()
    return int(year + month_day)


def discover_test_pairs(root):
    pair_directories = sorted(
        (path for path in Path(root).iterdir() if path.is_dir()),
        key=natural_pair_number,
    )
    if len(pair_directories) != 101:
        raise ValueError(f"Expected 101 TestData pairs, found {len(pair_directories)}")
    records = []
    for pair_directory in pair_directories:
        moving_files = list(pair_directory.glob("registered_masked_A_*.npz"))
        fixed_files = list(pair_directory.glob("fixed_masked_B_*.npz"))
        if len(moving_files) != 1 or len(fixed_files) != 1:
            raise ValueError(
                f"{pair_directory}: expected one moving and one fixed NPZ"
            )
        moving_path, fixed_path = moving_files[0], fixed_files[0]
        moving_patient = patient_id_from_name(moving_path)
        fixed_patient = patient_id_from_name(fixed_path)
        if moving_patient != fixed_patient:
            raise ValueError(f"Patient mismatch in {pair_directory}")
        if date_from_name(moving_path) >= date_from_name(fixed_path):
            raise ValueError(f"Expected earlier moving -> later fixed in {pair_directory}")
        records.append(
            TestPair(
                natural_pair_number(pair_directory),
                moving_patient,
                moving_path,
                fixed_path,
            )
        )
    return records


def load_volume(path):
    with np.load(path, allow_pickle=False) as archive:
        if "Train" not in archive.files:
            raise KeyError(f"Train key missing from {path}")
        volume = np.asarray(archive["Train"], dtype=np.float32)
    if volume.shape != EXPECTED_VOLUME_SHAPE:
        raise ValueError(f"{path}: shape {volume.shape}")
    if not np.isfinite(volume).all():
        raise ValueError(f"{path}: NaN/Inf detected")
    if float(volume.min()) < -1e-6 or float(volume.max()) > 1.0 + 1e-6:
        raise ValueError(f"{path}: range outside [0,1]")
    return np.ascontiguousarray(volume)


class TestPairDataset(Dataset):
    def __init__(self, records):
        self.records = list(records)

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        record = self.records[index]
        moving = torch.from_numpy(load_volume(record.moving_path)).unsqueeze(0)
        fixed = torch.from_numpy(load_volume(record.fixed_path)).unsqueeze(0)
        return moving, fixed, record.patient_id, record.pair_number


test_records = discover_test_pairs(TEST_ROOT)
test_patient_ids = {record.patient_id for record in test_records}
training_patient_ids = {
    patient_id_from_name(path)
    for path in LONGITUDINAL_ROOT.glob("pair*/*.npz")
}
overlap = sorted(test_patient_ids & training_patient_ids)
if overlap:
    raise ValueError(f"Training/TestData patient overlap: {overlap}")

test_loader = DataLoader(
    TestPairDataset(test_records),
    batch_size=1,
    shuffle=False,
    num_workers=0,
    pin_memory=device.type == "cuda",
)

print("Test pairs:", len(test_records))
print("Unique TestData patients:", len(test_patient_ids))
print("Longitudinal training/TestData overlap: 0")
print("This notebook performs inference only; no optimizer is created.")


## Final inference

The following cell evaluates all three checkpoints on the same 101 held-out pairs.
It can take several minutes. Volumes are loaded one pair at a time.


In [ ]:
def masked_body_metrics(fixed, prediction):
    mask = fixed > 0
    if int(mask.sum()) < 100:
        return float("nan"), float("nan"), float("nan")
    fixed_body = fixed[mask].float()
    prediction_body = prediction[mask].float()
    difference = fixed_body - prediction_body
    body_mse = torch.mean(difference ** 2)
    body_mae = torch.mean(torch.abs(difference))
    fixed_centered = fixed_body - fixed_body.mean()
    prediction_centered = prediction_body - prediction_body.mean()
    denominator = torch.sqrt(
        torch.sum(fixed_centered ** 2)
        * torch.sum(prediction_centered ** 2)
    ).clamp_min(1e-12)
    body_ncc = torch.sum(fixed_centered * prediction_centered) / denominator
    return float(body_mse), float(body_mae), float(body_ncc)


def eroded_fixed_body_mask(fixed):
    # One-voxel erosion keeps zero background and the body-mask edge from
    # dominating high-frequency/roughness measurements.
    mask = (fixed > 0).float()
    return -F.max_pool3d(-mask, kernel_size=3, stride=1, padding=1) > 0.5


def directional_difference_energy(image, mask, axis):
    first = [slice(None)] * image.ndim
    second = [slice(None)] * image.ndim
    first[axis] = slice(1, None)
    second[axis] = slice(None, -1)
    difference = image[tuple(first)] - image[tuple(second)]
    valid = mask[tuple(first)] & mask[tuple(second)]
    return torch.mean(difference[valid] ** 2).clamp_min(1e-12)


def texture_metrics(fixed, prediction):
    mask = eroded_fixed_body_mask(fixed)
    fixed_smooth = F.avg_pool3d(fixed, kernel_size=3, stride=1, padding=1)
    prediction_smooth = F.avg_pool3d(
        prediction, kernel_size=3, stride=1, padding=1
    )
    fixed_high = fixed - fixed_smooth
    prediction_high = prediction - prediction_smooth
    fixed_high_energy = torch.mean(fixed_high[mask] ** 2).clamp_min(1e-12)
    prediction_high_energy = torch.mean(prediction_high[mask] ** 2)
    high_frequency_ratio = torch.sqrt(
        prediction_high_energy / fixed_high_energy
    )
    high_frequency_residual = torch.sqrt(
        torch.mean((prediction_high[mask] - fixed_high[mask]) ** 2)
    )

    direction_names = (("z", 2), ("y", 3), ("x", 4))
    direction_ratios = {}
    fixed_direction_energies = []
    prediction_direction_energies = []
    for name, axis in direction_names:
        fixed_energy = directional_difference_energy(fixed, mask, axis)
        prediction_energy = directional_difference_energy(prediction, mask, axis)
        direction_ratios[f"difference_ratio_{name}"] = float(
            torch.sqrt(prediction_energy / fixed_energy)
        )
        fixed_direction_energies.append(fixed_energy)
        prediction_direction_energies.append(prediction_energy)
    total_variation_ratio = torch.sqrt(
        torch.stack(prediction_direction_energies).mean()
        / torch.stack(fixed_direction_energies).mean()
    )

    ratios = [high_frequency_ratio, total_variation_ratio] + [
        torch.as_tensor(value, device=fixed.device)
        for value in direction_ratios.values()
    ]
    texture_deviation = torch.stack(
        [torch.abs(torch.log(value.clamp_min(1e-12))) for value in ratios]
    ).mean()
    return {
        "high_frequency_ratio": float(high_frequency_ratio),
        "high_frequency_residual": float(high_frequency_residual),
        "total_variation_ratio": float(total_variation_ratio),
        "texture_deviation": float(texture_deviation),
        **direction_ratios,
    }


def zero_flow_reconstruction(moving):
    moving_bands = down_sampling_3d(analysis(moving))
    reconstructed, _ = synthesis_filter_3d(
        up_sampling_3d(moving_bands), synthesis_filters
    )
    return reconstructed


def flow_shape_metrics(flow):
    magnitude = torch.sqrt(torch.sum(flow ** 2, dim=1))
    second_difference_terms = []
    for axis in (2, 3, 4):
        first_difference = torch.diff(flow, dim=axis)
        second_difference_terms.append(
            torch.mean(torch.diff(first_difference, dim=axis) ** 2)
        )
    bending_energy = torch.stack(second_difference_terms).mean()

    # Sample every second voxel for topology diagnostics. This keeps the
    # 101-pair evaluation memory bounded while preserving large fold regions.
    full_shape = tuple(int(value) for value in flow.shape[2:])
    flow_np = (
        flow[0, :, ::2, ::2, ::2]
        .detach()
        .cpu()
        .numpy()
        .astype(np.float32)
    )
    gradients = [
        [
            np.gradient(
                flow_np[component], 2.0, axis=axis, edge_order=1
            )
            for axis in range(3)
        ]
        for component in range(3)
    ]
    jacobian = np.empty(flow_np.shape[1:] + (3, 3), dtype=np.float32)
    for component in range(3):
        for axis in range(3):
            jacobian[..., component, axis] = gradients[component][axis]
            if component == axis:
                jacobian[..., component, axis] += 1.0
    determinant = np.linalg.det(jacobian)
    positive = determinant[determinant > 0]

    coordinates = np.meshgrid(
        *[
            np.arange(size, dtype=np.float32) * 2.0
            for size in flow_np.shape[1:]
        ],
        indexing="ij",
    )
    outside = np.zeros(flow_np.shape[1:], dtype=bool)
    for axis, coordinate in enumerate(coordinates):
        displaced = coordinate + flow_np[axis]
        outside |= (displaced < 0) | (displaced > full_shape[axis] - 1)

    return {
        "flow_magnitude_p50": float(torch.quantile(magnitude, 0.50)),
        "flow_magnitude_p95": float(torch.quantile(magnitude, 0.95)),
        "flow_magnitude_max": float(magnitude.max()),
        "flow_bending_energy": float(bending_energy),
        "jacobian_fold_fraction": float(np.mean(determinant <= 0)),
        "log_jacobian_sd": (
            float(np.std(np.log(positive))) if positive.size else float("nan")
        ),
        "jacobian_extreme_fraction": float(
            np.mean((determinant < 0.5) | (determinant > 2.0))
        ),
        "out_of_bounds_fraction": float(outside.mean()),
    }


def evaluate_checkpoint(checkpoint_path, include_input_views=False):
    model = build_model(checkpoint_path)
    spatial_transformer = vxm.layers.SpatialTransformer(MODEL_SHAPE).to(device)
    results = {}
    views = {}

    with torch.inference_mode():
        for batch_index, (moving, fixed, patient_ids, pair_numbers) in enumerate(test_loader, 1):
            moving = moving.to(device, non_blocking=True)
            fixed = fixed.to(device, non_blocking=True)
            transformed, flow = registration_forward(
                model, spatial_transformer, moving, fixed
            )

            whole_mse = torch.mean((fixed - transformed) ** 2)
            smoothness = flow_gradient_loss(None, flow)
            total_loss = whole_mse + GRAD_WEIGHT * smoothness
            rigid_body_mse, rigid_body_mae, rigid_body_ncc = masked_body_metrics(
                fixed, moving
            )
            body_mse, body_mae, body_ncc = masked_body_metrics(
                fixed, transformed
            )
            transformed_texture = texture_metrics(fixed, transformed)
            flow_metrics = flow_shape_metrics(flow)
            pair_number = int(pair_numbers[0])
            patient_id = patient_ids[0]
            results[pair_number] = {
                "pair_number": pair_number,
                "patient_id": patient_id,
                "rigid_whole_mse": float(torch.mean((fixed - moving) ** 2)),
                "rigid_body_mse": rigid_body_mse,
                "rigid_body_mae": rigid_body_mae,
                "rigid_body_ncc": rigid_body_ncc,
                "total_loss": float(total_loss),
                "whole_mse": float(whole_mse),
                "body_mse": body_mse,
                "body_mae": body_mae,
                "body_ncc": body_ncc,
                "flow_gradient": float(smoothness),
                **transformed_texture,
                **flow_metrics,
            }

            if include_input_views:
                rigid_texture = texture_metrics(fixed, moving)
                zero_reconstruction = zero_flow_reconstruction(moving)
                zero_body_mse, zero_body_mae, zero_body_ncc = masked_body_metrics(
                    fixed, zero_reconstruction
                )
                zero_fixed_texture = texture_metrics(fixed, zero_reconstruction)
                zero_moving_texture = texture_metrics(moving, zero_reconstruction)
                results[pair_number].update(
                    {
                        **{f"rigid_{key}": value for key, value in rigid_texture.items()},
                        "zero_reconstruction_mse_vs_moving": float(
                            torch.mean((zero_reconstruction - moving) ** 2)
                        ),
                        "zero_whole_mse": float(
                            torch.mean((fixed - zero_reconstruction) ** 2)
                        ),
                        "zero_body_mse": zero_body_mse,
                        "zero_body_mae": zero_body_mae,
                        "zero_body_ncc": zero_body_ncc,
                        **{
                            f"zero_{key}": value
                            for key, value in zero_fixed_texture.items()
                        },
                        **{
                            f"zero_vs_moving_{key}": value
                            for key, value in zero_moving_texture.items()
                        },
                    }
                )

            central_index = fixed.shape[2] // 2
            views[pair_number] = {
                "warped": transformed[0, 0, central_index].cpu().numpy()
            }
            if include_input_views:
                views[pair_number].update(
                    {
                        "fixed": fixed[0, 0, central_index].cpu().numpy(),
                        "rigid_moving": moving[0, 0, central_index].cpu().numpy(),
                        "zero_reconstruction": zero_reconstruction[
                            0, 0, central_index
                        ].cpu().numpy(),
                    }
                )
            if batch_index % 10 == 0 or batch_index == len(test_loader):
                print(
                    f"{Path(checkpoint_path).stem}: "
                    f"{batch_index}/{len(test_loader)} pairs"
                )

    del model, spatial_transformer
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return results, views


for checkpoint in (PRETRAIN_PATH, DIFFERENT_PATIENT_PATH, FINETUNED_PATH):
    if not checkpoint.is_file():
        raise FileNotFoundError(checkpoint)

pretrained_results, pretrained_views = evaluate_checkpoint(
    PRETRAIN_PATH, include_input_views=True
)
different_patient_results, different_patient_views = evaluate_checkpoint(
    DIFFERENT_PATIENT_PATH, include_input_views=False
)
finetuned_results, finetuned_views = evaluate_checkpoint(
    FINETUNED_PATH, include_input_views=False
)
print("Inference complete for all three checkpoints.")


In [ ]:
MODEL_KEYS = [
    "total_loss", "whole_mse", "body_mse", "body_mae", "body_ncc",
    "flow_gradient", "high_frequency_ratio", "high_frequency_residual",
    "total_variation_ratio", "texture_deviation", "difference_ratio_z",
    "difference_ratio_y", "difference_ratio_x", "flow_magnitude_p50",
    "flow_magnitude_p95", "flow_magnitude_max", "flow_bending_energy",
    "jacobian_fold_fraction", "log_jacobian_sd",
    "jacobian_extreme_fraction", "out_of_bounds_fraction",
]

REFERENCE_KEYS = [
    "rigid_whole_mse", "rigid_body_mse", "rigid_body_mae", "rigid_body_ncc",
    "rigid_high_frequency_ratio", "rigid_high_frequency_residual",
    "rigid_total_variation_ratio", "rigid_texture_deviation",
    "rigid_difference_ratio_z", "rigid_difference_ratio_y",
    "rigid_difference_ratio_x", "zero_reconstruction_mse_vs_moving",
    "zero_whole_mse", "zero_body_mse", "zero_body_mae", "zero_body_ncc",
    "zero_high_frequency_ratio", "zero_high_frequency_residual",
    "zero_total_variation_ratio", "zero_texture_deviation",
    "zero_difference_ratio_z", "zero_difference_ratio_y", "zero_difference_ratio_x",
    "zero_vs_moving_high_frequency_ratio",
    "zero_vs_moving_high_frequency_residual",
    "zero_vs_moving_total_variation_ratio",
    "zero_vs_moving_texture_deviation",
    "zero_vs_moving_difference_ratio_z", "zero_vs_moving_difference_ratio_y",
    "zero_vs_moving_difference_ratio_x",
]

rows = []
for record in test_records:
    curriculum = pretrained_results[record.pair_number]
    different = different_patient_results[record.pair_number]
    same = finetuned_results[record.pair_number]
    row = {"pair_number": record.pair_number, "patient_id": record.patient_id}
    row.update({key: curriculum[key] for key in REFERENCE_KEYS})
    for prefix, result in (
        ("curriculum", curriculum),
        ("different_patient", different),
        ("same_patient", same),
    ):
        row.update({f"{prefix}_{key}": result[key] for key in MODEL_KEYS})
    rows.append(row)

pair_metrics = pd.DataFrame(rows).sort_values("pair_number").reset_index(drop=True)
for metric in (
    "whole_mse", "body_mse", "body_ncc", "high_frequency_residual",
    "texture_deviation", "flow_gradient", "flow_bending_energy",
    "jacobian_fold_fraction", "out_of_bounds_fraction",
):
    pair_metrics[f"{metric}_delta_same_minus_different"] = (
        pair_metrics[f"same_patient_{metric}"]
        - pair_metrics[f"different_patient_{metric}"]
    )
pair_metrics.to_csv(OUTPUT_ROOT / "roughness_pair_metrics.csv", index=False)

numeric_columns = pair_metrics.select_dtypes(include=[np.number]).columns
patient_metrics = (
    pair_metrics.groupby("patient_id", as_index=False)[numeric_columns]
    .mean()
    .drop(columns=["pair_number"])
)
patient_metrics.to_csv(OUTPUT_ROOT / "roughness_patient_macro_metrics.csv", index=False)


def method_means(frame, prefix, model_method):
    image_keys = [
        "whole_mse", "body_mse", "body_mae", "body_ncc",
        "high_frequency_ratio", "high_frequency_residual",
        "total_variation_ratio", "texture_deviation", "difference_ratio_z",
        "difference_ratio_y", "difference_ratio_x",
    ]
    result = {
        key: float(frame[f"{prefix}_{key}"].mean()) for key in image_keys
    }
    if model_method:
        for key in (
            "total_loss", "flow_gradient", "flow_magnitude_p50",
            "flow_magnitude_p95", "flow_magnitude_max", "flow_bending_energy",
            "jacobian_fold_fraction", "log_jacobian_sd",
            "jacobian_extreme_fraction", "out_of_bounds_fraction",
        ):
            result[key] = float(frame[f"{prefix}_{key}"].mean())
    return result


def all_method_summary(frame):
    return {
        "rigid_input": method_means(frame, "rigid", False),
        "zero_flow_reconstruction": method_means(frame, "zero", False),
        "curriculum_only": method_means(frame, "curriculum", True),
        "old_different_patient_finetune": method_means(
            frame, "different_patient", True
        ),
        "new_same_patient_finetune": method_means(frame, "same_patient", True),
    }


pair_summary = all_method_summary(pair_metrics)
patient_summary = all_method_summary(patient_metrics)


def percent_reduction(before, after):
    return 100.0 * (before - after) / before


def patient_bootstrap_ci(delta_column, repetitions=10000, seed=20260717):
    values = patient_metrics[delta_column].to_numpy(dtype=np.float64)
    generator = np.random.default_rng(seed)
    sampled_means = np.empty(repetitions, dtype=np.float64)
    for index in range(repetitions):
        sampled_means[index] = generator.choice(
            values, size=len(values), replace=True
        ).mean()
    return [
        float(values.mean()),
        float(np.percentile(sampled_means, 2.5)),
        float(np.percentile(sampled_means, 97.5)),
    ]


PRIMARY_DELTAS = {
    metric: patient_bootstrap_ci(f"{metric}_delta_same_minus_different")
    for metric in (
        "whole_mse", "body_mse", "body_ncc", "high_frequency_residual",
        "texture_deviation", "flow_gradient", "flow_bending_energy",
        "jacobian_fold_fraction", "out_of_bounds_fraction",
    )
}

WILCOXON_P = {
    metric: float(
        wilcoxon(
            patient_metrics[f"same_patient_{metric}"],
            patient_metrics[f"different_patient_{metric}"],
            alternative="two-sided",
        ).pvalue
    )
    for metric in (
        "whole_mse", "body_mse", "body_ncc", "high_frequency_residual",
        "texture_deviation", "flow_gradient", "flow_bending_energy",
    )
}

IMPROVEMENT_COUNTS = {
    "whole_mse_lower": int(
        (pair_metrics["same_patient_whole_mse"] < pair_metrics["different_patient_whole_mse"]).sum()
    ),
    "body_mse_lower": int(
        (pair_metrics["same_patient_body_mse"] < pair_metrics["different_patient_body_mse"]).sum()
    ),
    "body_ncc_higher": int(
        (pair_metrics["same_patient_body_ncc"] > pair_metrics["different_patient_body_ncc"]).sum()
    ),
    "high_frequency_residual_lower": int(
        (pair_metrics["same_patient_high_frequency_residual"] < pair_metrics["different_patient_high_frequency_residual"]).sum()
    ),
    "texture_deviation_lower": int(
        (pair_metrics["same_patient_texture_deviation"] < pair_metrics["different_patient_texture_deviation"]).sum()
    ),
    "flow_bending_energy_lower": int(
        (pair_metrics["same_patient_flow_bending_energy"] < pair_metrics["different_patient_flow_bending_energy"]).sum()
    ),
}

old_summary = patient_summary["old_different_patient_finetune"]
new_summary = patient_summary["new_same_patient_finetune"]
summary = {
    "purpose": "diagnose rough output from old different-patient fine-tuning",
    "test_pairs": int(len(pair_metrics)),
    "unique_test_patients": int(len(patient_metrics)),
    "training_test_patient_overlap": [],
    "pair_macro": pair_summary,
    "patient_macro_primary": patient_summary,
    "patient_macro_same_vs_different_reduction_percent": {
        metric: percent_reduction(old_summary[metric], new_summary[metric])
        for metric in (
            "whole_mse", "body_mse", "body_mae",
            "high_frequency_residual", "texture_deviation",
            "flow_gradient", "flow_bending_energy",
        )
    },
    "patient_cluster_bootstrap_mean_delta_same_minus_different_and_95pct_ci": PRIMARY_DELTAS,
    "patient_level_two_sided_wilcoxon_p": WILCOXON_P,
    "pair_improvement_counts_out_of_101": IMPROVEMENT_COUNTS,
    "zero_flow_control": {
        "mse_vs_original_moving": float(
            patient_metrics["zero_reconstruction_mse_vs_moving"].mean()
        ),
        "high_frequency_ratio_vs_original_moving": float(
            patient_metrics["zero_vs_moving_high_frequency_ratio"].mean()
        ),
        "total_variation_ratio_vs_original_moving": float(
            patient_metrics["zero_vs_moving_total_variation_ratio"].mean()
        ),
    },
    "causal_caveat": (
        "Old and new fine-tuning differ in pairing, learning rate, epoch count, "
        "early stopping, and flow-gradient regularization; this comparison "
        "supports a hypothesis but does not isolate pairing alone."
    ),
}
(OUTPUT_ROOT / "roughness_summary.json").write_text(
    json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8"
)

DISPLAY_COLUMNS = [
    "whole_mse", "body_mse", "body_ncc", "high_frequency_ratio",
    "high_frequency_residual", "total_variation_ratio", "texture_deviation",
    "flow_gradient", "flow_bending_energy", "jacobian_fold_fraction",
    "out_of_bounds_fraction",
]
print("PAIR-MACRO (101 pairs; repeated patients are weighted repeatedly)")
display(pd.DataFrame(pair_summary).T.reindex(columns=DISPLAY_COLUMNS))
print("PATIENT-MACRO (45 patients; use this as the primary summary)")
display(pd.DataFrame(patient_summary).T.reindex(columns=DISPLAY_COLUMNS))
print("Same-patient minus different-patient [patient mean delta, 95% cluster CI]:")
for metric, interval in PRIMARY_DELTAS.items():
    print(f" {metric}: {interval}")
print("Pair improvement counts out of 101:", IMPROVEMENT_COUNTS)
print("Zero-flow reconstruction control:", summary["zero_flow_control"])
print("IMPORTANT:", summary["causal_caveat"])
print("Saved:", OUTPUT_ROOT / "roughness_pair_metrics.csv")
print("Saved:", OUTPUT_ROOT / "roughness_patient_macro_metrics.csv")
print("Saved:", OUTPUT_ROOT / "roughness_summary.json")


In [ ]:
# Show both the strongest roughness improvements and the least-improved cases.
from scipy.ndimage import uniform_filter


def high_pass_2d(image):
    return image - uniform_filter(image, size=3, mode="nearest")


most_improved = pair_metrics.nsmallest(
    3, "texture_deviation_delta_same_minus_different"
)
least_improved = pair_metrics.nlargest(
    3, "texture_deviation_delta_same_minus_different"
)
selected_pairs = pd.concat([most_improved, least_improved]).drop_duplicates(
    subset=["pair_number"]
)

for row in selected_pairs.itertuples(index=False):
    pair_number = int(row.pair_number)
    patient_id = row.patient_id
    fixed = pretrained_views[pair_number]["fixed"]
    rigid_moving = pretrained_views[pair_number]["rigid_moving"]
    zero_reconstruction = pretrained_views[pair_number]["zero_reconstruction"]
    curriculum_warped = pretrained_views[pair_number]["warped"]
    different_warped = different_patient_views[pair_number]["warped"]
    same_warped = finetuned_views[pair_number]["warped"]
    different_error = np.abs(fixed - different_warped)
    same_error = np.abs(fixed - same_warped)
    error_max = max(
        float(np.percentile(different_error, 99)),
        float(np.percentile(same_error, 99)),
        1e-4,
    )

    figure, axes = plt.subplots(3, 4, figsize=(16, 12))
    grayscale = [
        (fixed, "Fixed (later)"),
        (rigid_moving, "Rigid moving (earlier)"),
        (zero_reconstruction, "Zero-flow reconstruction"),
        (curriculum_warped, "Curriculum only"),
        (different_warped, "Old: different-patient FT"),
        (same_warped, "New: same-patient FT"),
    ]
    for axis, (image, title) in zip(axes.flat[:6], grayscale):
        axis.imshow(image, cmap="gray", vmin=0.0, vmax=1.0)
        axis.set_title(title)
        axis.axis("off")
    axes[1, 2].imshow(different_error, cmap="magma", vmin=0, vmax=error_max)
    axes[1, 2].set_title("|Fixed - Different-patient FT|")
    axes[1, 2].axis("off")
    axes[1, 3].imshow(same_error, cmap="magma", vmin=0, vmax=error_max)
    axes[1, 3].set_title("|Fixed - Same-patient FT|")
    axes[1, 3].axis("off")

    high_pass_images = [
        (high_pass_2d(fixed), "High-pass: Fixed"),
        (high_pass_2d(curriculum_warped), "High-pass: Curriculum"),
        (high_pass_2d(different_warped), "High-pass: Different FT"),
        (high_pass_2d(same_warped), "High-pass: Same FT"),
    ]
    high_limit = max(
        float(np.percentile(np.abs(image), 99))
        for image, _ in high_pass_images
    )
    for axis, (image, title) in zip(axes[2], high_pass_images):
        axis.imshow(image, cmap="coolwarm", vmin=-high_limit, vmax=high_limit)
        axis.set_title(title)
        axis.axis("off")
    figure.suptitle(
        f"Test pair {pair_number}: {patient_id} | "
        "texture deviation delta same-different="
        f"{row.texture_deviation_delta_same_minus_different:+.6f}"
    )
    figure.tight_layout()
    output_path = OUTPUT_ROOT / f"pair{pair_number}_{patient_id}_roughness.png"
    figure.savefig(output_path, dpi=150, bbox_inches="tight")
    plt.show()

figure, axes = plt.subplots(1, 2, figsize=(12, 5))
for axis, metric, title in (
    (axes[0], "body_mse", "Body MSE (lower is better)"),
    (axes[1], "texture_deviation", "Texture deviation (lower is closer to Fixed)"),
):
    x = patient_metrics[f"different_patient_{metric}"]
    y = patient_metrics[f"same_patient_{metric}"]
    axis.scatter(x, y, alpha=0.8)
    limit = float(max(x.max(), y.max()))
    axis.plot([0, limit], [0, limit], "k--", linewidth=1)
    axis.set_xlim(0, limit)
    axis.set_ylim(0, limit)
    axis.set_aspect("equal", adjustable="box")
    axis.set_xlabel("Old different-patient fine-tune")
    axis.set_ylabel("New same-patient fine-tune")
    axis.set_title(title)
    axis.grid(True, alpha=0.3)
figure.tight_layout()
figure.savefig(OUTPUT_ROOT / "patient_old_vs_new_scatter.png", dpi=160)
plt.show()

print("Visual QC files saved under:", OUTPUT_ROOT)


## Interpretation

Use the 45-patient macro result as the primary summary because 14 patients have
multiple dependent pairs. For same-minus-different deltas, negative MSE,
high-frequency residual, texture deviation, and flow bending favor the new
same-patient model; positive NCC favors it. Ratios near 1 mean that the output
texture is close to Fixed, values above 1 suggest excess high-frequency content,
and values below 1 suggest smoothing.

This is a diagnostic comparison, not a clean causal proof: the old and new runs
also differ in learning rate, epoch count, early stopping, and flow-gradient
regularization. A controlled follow-up must keep those settings identical and
change only the patient-pairing rule.
